In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import sum as _sum, count, round as _round
from delta.tables import DeltaTable

silver_df = spark.table(silver_table)
 
gold_agg_df = (
    silver_df.groupBy("user_id")
    .agg(
        _round(_sum("amount"), 2).alias("total_spent"),
        count("transaction_id").alias("transaction_count"),
    )
)
 
if not spark.catalog.tableExists(gold_table):
    gold_agg_df.write.format("delta").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    (gold_delta.alias("target")
        .merge(
            source=gold_agg_df.alias("g"),
            condition="target.user_id = g.user_id",
        )
        .whenMatchedUpdate(set={
            "total_spent": "g.total_spent",
            "transaction_count": "g.transaction_count",
        })
        .whenNotMatchedInsert(values={
            "user_id": "g.user_id",
            "total_spent": "g.total_spent",
            "transaction_count": "g.transaction_count",
        })
        .execute())
 
display(spark.table(gold_table).orderBy("user_id"))